# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [2]:
!pip install -U langchain langchain-openai langchain-community



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully 

In [3]:
!pip install langgraph


In [4]:
from typing import Optional

from langchain_core.runnables import RunnableSequence
from langgraph.prebuilt import create_react_agent
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [5]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [6]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [7]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 25.4 MB/s eta 0:00:00


In [8]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings(openai_api_key="APIKEY")
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})
# Helper functions for memory
def add_to_memory(task: str, result: str):
    vectorstore.add_texts([result], metadatas=[{"task": task}])

def retrieve_context(query: str, k: int = 3) -> str:
    docs = vectorstore.similarity_search(query, k=k)
    return "\n".join([doc.page_content for doc in docs])

In [9]:
!pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=01b69bf8b12a71e23a3c362efa88d797a00f5651d045a7420437c193bd36034e
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [22]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import PromptTemplate
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import ChatOpenAI
execution_prompt = PromptTemplate(
    input_variables=["objective", "task", "context"],
    template="""
You are an AI executor.
Objective: {objective}
Task: {task}
Context: {context}

Provide a clear result.
"""
)

task_creation_prompt = PromptTemplate(
    input_variables=["objective", "result", "task_list"],
    template="""
You are a task creation AI.
Objective: {objective}
The result of the last task: {result}
Incomplete tasks: {task_list}

Create new tasks (one per line).
"""
)

task_prioritization_prompt = PromptTemplate(
    input_variables=["objective", "task_list"],
    template="""
You are a task prioritization AI.
Objective: {objective}
Reprioritize these tasks in order of importance:
{task_list}
Return as a numbered list.
"""
)


In [36]:

llm=ChatOpenAI(temperature=0,openai_api_key="API_KEY")
# todo_chain = RunnableSequence([todo_prompt,llm])
execution_chain = RunnableSequence(first=execution_prompt,middle=[], last=llm)
task_creation_chain = RunnableSequence(first=task_creation_prompt,middle=[], last=llm)
task_prioritization_chain = RunnableSequence(first=task_prioritization_prompt,middle=[], last=llm)


def todo_tool(objective: str, last_result: str = "", task_list_text: str = "") -> str:
    """
    Generates new tasks based on objective, last result, and incomplete tasks.
    Returns tasks as string (one per line).
    """
    output = task_creation_chain.invoke({
        "objective": objective,
        "result": last_result,
        "task_list": task_list_text
    })
    # Convert AIMessage to string if necessary
    if hasattr(output, "content"):
        return output.content
    return str(output)

search = SerpAPIWrapper(serpapi_api_key="APIKEY")

def search_tool(query: str) -> str:
    """
    Searches the web using SerpAPI and returns relevant results.
    """
    return search.run(query)



In [37]:
tools = [todo_tool, search_tool]

agent = create_agent(
    llm,
    tools=tools,
    system_prompt="You are an autonomous AI agent. Use tools wisely."
)

In [38]:
from collections import deque
def babyagi_controller(objective: str, max_iterations: int = 5):
    task_list = deque(["Start initial research"])
    completed_tasks = []

    for _ in range(max_iterations):
        if not task_list:
            break

        task = task_list.popleft()

        # Retrieve relevant memory
        context = retrieve_context(task)

        # Execute task
        output = execution_chain.invoke({
            "objective": objective,
            "task": task,
            "context": context
        })
        if hasattr(output, "content"):
             result = output.content
        else:
             result = str(output)
        completed_tasks.append(f"{task}: {result}")

        # Store result in vectorstore memory
        add_to_memory(task, result)

        # Generate new tasks
        new_tasks_text = todo_tool(
            objective,
            last_result=result,
            task_list_text="\n".join(task_list)
        )
        if hasattr(new_tasks_text, "content"):
             new_tasks_text = new_tasks_text.content
        else:
             new_tasks_text = str(new_tasks_text)
        for t in new_tasks_text.split("\n"):
            if t.strip():
                task_list.append(t.strip())

        # Re-prioritize tasks
        if task_list:
            prioritized_text = task_prioritization_chain.invoke({
                "objective": objective,
                "task_list": "\n".join(task_list)
            })
            task_list = deque([t.split(". ",1)[1] for t in prioritized_text.split("\n") if t.strip()])

    return completed_tasks


In [39]:
objective = "Build a list of AI startups in 2025"

completed = babyagi_controller(objective, max_iterations=5)

print("Completed Tasks:\n")
for task in completed:
    print(task)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [12]:
OBJECTIVE = "Write a weather report for SF today"

In [13]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

NameError: name 'BabyAGI' is not defined

In [ ]:
from collections import deque

def babyagi_controller(objective: str, max_iterations: int = 5):
    """
    Runs a BabyAGI-style loop with one agent, two tools, and memory.
    """
    task_list = deque(["Start initial research"])
    completed_tasks = []

    for _ in range(max_iterations):
        if not task_list:
            break

        task = task_list.popleft()

        # Retrieve context from memory
        context = retrieve_context(task)

        # Execute task
        result = execution_chain.invoke({
            "objective": objective,
            "task": task,
            "context": context
        })
        completed_tasks.append(f"{task}: {result}")

        # Store result in vectorstore memory
        add_to_memory(task, result)

        # Generate new tasks
        new_tasks_text = todo_tool(
            objective,
            last_result=result,
            task_list_text="\n".join(task_list)
        )
        for t in new_tasks_text.split("\n"):
            if t.strip():
                task_list.append(t.strip())

        # Re-prioritize tasks
        if task_list:
            prioritized_text = task_prioritization_chain.invoke({
                "objective": objective,
                "task_list": "\n".join(task_list)
            })
            task_li_


In [ ]:
baby_agi({"objective": OBJECTIVE})